In [1]:
import pandas as pd
import configure_lichess_csv as cfg
import chess
import chess.engine
from stockfish import Stockfish
import dask.dataframe as dd
import swifter

In [17]:
data = pd.read_json("chess_data_20K_evals.jsonl", lines=True)

In [18]:
data.shape

(20003, 3)

In [19]:
data.head()

,instruction,input,output
0,Select the best move for this position. Please...,FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKB...,<uci_move>e2e4</uci_move>
1,Select the best move for this position. Please...,FEN: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQ...,<uci_move>c7c5</uci_move>
2,Select the best move for this position. Please...,FEN: rnbqk2r/ppp2ppp/3b4/8/2P3n1/5N1P/PP2PPP1/...,<uci_move>g4f2</uci_move>
3,Select the best move for this position. Please...,FEN: 4r3/1k6/pp3r2/1b2P2p/3R1p2/P1R2P2/1P4PP/6...,<uci_move>a3a4</uci_move>
4,Select the best move for this position. Please...,FEN: r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2R1/PqP2bPP/7K...,<uci_move>b2b1</uci_move>


In [20]:
data.iloc[0]["input"]

"FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1\nLegalMoves:  a2a3, a2a4, b1a3, b1c3, b2b3, b2b4, c2c3, c2c4, d2d3, d2d4, e2e3, e2e4, f2f3, f2f4, g1f3, g1h3, g2g3, g2g4, h2h3, h2h4\nLogicalMoves[{'Move': 'e2e4', 'Centipawn': 30, 'Mate': None}, {'Move': 'd2d4', 'Centipawn': 23, 'Mate': None}, {'Move': 'g1f3', 'Centipawn': 18, 'Mate': None}, {'Move': 'e2e3', 'Centipawn': 14, 'Mate': None}, {'Move': 'c2c4', 'Centipawn': 12, 'Mate': None}]"

In [27]:
data["logical_moves"] = data["input"].str.split("\nLogicalMoves").str.get(1)

In [40]:
data["original_input"] = data["input"].str.split("\nLogicalMoves").str.get(0)

In [32]:
data.iloc[1]["logical_moves"].str.split("{")

"[{'Move': 'c7c5', 'Centipawn': 31, 'Mate': None}, {'Move': 'e7e6', 'Centipawn': 32, 'Mate': None}, {'Move': 'e7e5', 'Centipawn': 37, 'Mate': None}, {'Move': 'c7c6', 'Centipawn': 39, 'Mate': None}, {'Move': 'd7d6', 'Centipawn': 43, 'Mate': None}]"

Convert top moves into a readable format. 

In [37]:
import ast

data["formatted_moves"] = data["logical_moves"].apply(
    lambda s: "Legal Moves with Highest Centipawn Scores (cp): " + ", ".join(
        f"{m['Move']} ({m['Centipawn']} cp)"
        for m in ast.literal_eval(s)
    )
)


In [38]:
data.iloc[0]["formatted_moves"]

'Legal Moves with Highest Centipawn Scores (cp): e2e4 (30 cp), d2d4 (23 cp), g1f3 (18 cp), e2e3 (14 cp), c2c4 (12 cp)'

In [42]:
data.iloc[0]["original_input"]

'FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1\nLegalMoves:  a2a3, a2a4, b1a3, b1c3, b2b3, b2b4, c2c3, c2c4, d2d3, d2d4, e2e3, e2e4, f2f3, f2f4, g1f3, g1h3, g2g3, g2g4, h2h3, h2h4'

We split the string in the second column to obtain the raw FEN data. 

In [9]:
# first split the legal moves from the fen 
data["fen_input"] = data["input"].str.split("\n").str.get(0)

In [48]:
data["Legal Moves"] = data["original_input"].str.split("\nLegalMoves: ").str.get(1)

In [50]:
data["FEN"] = data["original_input"].str.split("\nLegalMoves: ").str.get(0)

In [51]:
data.iloc[0]["FEN"]

'FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'

In [70]:
data["instruction"] = "Select the best move for this position. The best move must be selected from the set of Legal Moves with Highest Centipawn Scores (cp). For example, if a1d2 (50 cp) is within the subset of best legal moves and the most logical choice, return <uci_move>a1d2</uci_move>. Please justify your reasoning in 1 or 2 sentences."

In [71]:
data.columns

Index(['instruction', 'input', 'output', 'logical_moves', 'formatted_moves',
       'original_input', 'Legal Moves', 'FEN'],
      dtype='object')

In [66]:
data["input"] = (data["FEN"].astype(str) + 
                 "\nLegal Moves: " + data["Legal Moves"] + 
                 "\n" + data["formatted_moves"].astype(str))

In [69]:
data.iloc[0]["input"]

'FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1\nLegal Moves:  a2a3, a2a4, b1a3, b1c3, b2b3, b2b4, c2c3, c2c4, d2d3, d2d4, e2e3, e2e4, f2f3, f2f4, g1f3, g1h3, g2g3, g2g4, h2h3, h2h4\nLegal Moves with Highest Centipawn Scores (cp): e2e4 (30 cp), d2d4 (23 cp), g1f3 (18 cp), e2e3 (14 cp), c2c4 (12 cp)'

In [72]:
data

,instruction,input,output,logical_moves,formatted_moves,original_input,Legal Moves,FEN
0,Select the best move for this position. The be...,FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKB...,<uci_move>e2e4</uci_move>,"[{'Move': 'e2e4', 'Centipawn': 30, 'Mate': Non...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKB...,"a2a3, a2a4, b1a3, b1c3, b2b3, b2b4, c2c3, c2c...",FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKB...
1,Select the best move for this position. The be...,FEN: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQ...,<uci_move>c7c5</uci_move>,"[{'Move': 'c7c5', 'Centipawn': 31, 'Mate': Non...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQ...,"g8h6, g8f6, b8c6, b8a6, h7h6, g7g6, f7f6, e7e...",FEN: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQ...
2,Select the best move for this position. The be...,FEN: rnbqk2r/ppp2ppp/3b4/8/2P3n1/5N1P/PP2PPP1/...,<uci_move>g4f2</uci_move>,"[{'Move': 'g4f2', 'Centipawn': -307, 'Mate': N...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: rnbqk2r/ppp2ppp/3b4/8/2P3n1/5N1P/PP2PPP1/...,"h8g8, h8f8, e8f8, e8e7, e8d7, d8e7, d8d7, d8f...",FEN: rnbqk2r/ppp2ppp/3b4/8/2P3n1/5N1P/PP2PPP1/...
3,Select the best move for this position. The be...,FEN: 4r3/1k6/pp3r2/1b2P2p/3R1p2/P1R2P2/1P4PP/6...,<uci_move>a3a4</uci_move>,"[{'Move': 'a3a4', 'Centipawn': 60, 'Mate': Non...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: 4r3/1k6/pp3r2/1b2P2p/3R1p2/P1R2P2/1P4PP/6...,"d4d8, d4d7, d4d6, d4d5, d4f4, d4e4, d4c4, d4b...",FEN: 4r3/1k6/pp3r2/1b2P2p/3R1p2/P1R2P2/1P4PP/6...
4,Select the best move for this position. The be...,FEN: r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2R1/PqP2bPP/7K...,<uci_move>b2b1</uci_move>,"[{'Move': 'f2e3', 'Centipawn': -503, 'Mate': N...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2R1/PqP2bPP/7K...,"a8g8, a8f8, a8e8, a8d8, a8c8, a8b8, e7e8, e7g...",FEN: r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2R1/PqP2bPP/7K...
...,...,...,...,...,...,...,...,...
19998,Select the best move for this position. The be...,FEN: 5k2/1p2pp2/p1q2r2/6Rp/2pP2P1/N1Pb1PP1/PP3...,<uci_move>g5c5</uci_move>,"[{'Move': 'd4d5', 'Centipawn': 28, 'Mate': Non...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: 5k2/1p2pp2/p1q2r2/6Rp/2pP2P1/N1Pb1PP1/PP3...,nan,FEN: 5k2/1p2pp2/p1q2r2/6Rp/2pP2P1/N1Pb1PP1/PP3...
19999,Select the best move for this position. The be...,FEN: r3r1k1/pp3ppp/3R1q2/n7/2Q5/1B3P2/PPP2P1P/...,<uci_move>a5b3</uci_move>,"[{'Move': 'a5b3', 'Centipawn': -575, 'Mate': N...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: r3r1k1/pp3ppp/3R1q2/n7/2Q5/1B3P2/PPP2P1P/...,nan,FEN: r3r1k1/pp3ppp/3R1q2/n7/2Q5/1B3P2/PPP2P1P/...
20000,Select the best move for this position. The be...,FEN: 8/1p6/3k2p1/6P1/2PK4/8/8/8 b - - 0 45\nd6...,<uci_move>b7b6</uci_move>,"[{'Move': 'b7b6', 'Centipawn': 0, 'Mate': None...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: 8/1p6/3k2p1/6P1/2PK4/8/8/8 b - - 0 45\nd6...,nan,FEN: 8/1p6/3k2p1/6P1/2PK4/8/8/8 b - - 0 45\nd6...
20001,Select the best move for this position. The be...,FEN: 8/8/b7/1p1p1kRp/1p3P2/2r1PN2/P2K4/8 b - -...,<uci_move>f5f6</uci_move>,"[{'Move': 'f5f6', 'Centipawn': 16, 'Mate': Non...",Legal Moves with Highest Centipawn Scores (cp)...,FEN: 8/8/b7/1p1p1kRp/1p3P2/2r1PN2/P2K4/8 b - -...,nan,FEN: 8/8/b7/1p1p1kRp/1p3P2/2r1PN2/P2K4/8 b - -...


In [22]:
data[["instruction", "input", "output"]].to_json(
    "chess_data_labeled.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)